# Olist PySpark ETL
Clean project notebook containing only the code needed to run and verify the pipeline.


In [132]:
from pyspark.sql import SparkSession
from pyspark.sql.types import (
    StructType, StructField, StringType, TimestampType,
    IntegerType, DoubleType, DecimalType
)

spark = (
    SparkSession.builder
    .appName("OlistETL")
    .master("local[*]")
    .getOrCreate()
)

spark.version


'4.1.1'

## Schemas


In [133]:
order_schema = StructType([
    StructField("order_id", StringType()),
    StructField("customer_id", StringType()),
    StructField("order_status", StringType()),
    StructField("order_purchase_timestamp", TimestampType()),
    StructField("order_approved_at", TimestampType()),
    StructField("order_delivered_carrier_date", TimestampType()),
    StructField("order_delivered_customer_date", TimestampType()),
    StructField("order_estimated_delivery_date", TimestampType())
])

order_item_schema = StructType([
    StructField("order_id", StringType()),
    StructField("order_item_id", IntegerType()),
    StructField("product_id", StringType()),
    StructField("seller_id", StringType()),
    StructField("shipping_limit_date", TimestampType()),
    StructField("price", DoubleType()),
    StructField("freight_value", DoubleType())
])

customer_schema = StructType([
    StructField("customer_id", StringType()),
    StructField("customer_unique_id", StringType()),
    StructField("customer_zip_code_prefix", StringType()),
    StructField("customer_city", StringType()),
    StructField("customer_state", StringType())
])

product_schema = StructType([
    StructField("product_id", StringType()),
    StructField("product_category_name", StringType()),
    StructField("product_name_lenght", IntegerType()),
    StructField("product_description_lenght", IntegerType()),
    StructField("product_photos_qty", IntegerType()),
    StructField("product_weight_g", IntegerType()),
    StructField("product_length_cm", IntegerType()),
    StructField("product_height_cm", IntegerType()),
    StructField("product_width_cm", IntegerType())
])

seller_schema = StructType([
    StructField("seller_id", StringType()),
    StructField("seller_zip_code_prefix", StringType()),
    StructField("seller_city", StringType()),
    StructField("seller_state", StringType())
])

payment_schema = StructType([
    StructField("order_id", StringType()),
    StructField("payment_sequential", IntegerType()),
    StructField("payment_type", StringType()),
    StructField("payment_installments", IntegerType()),
    StructField("payment_value", DecimalType(18, 2))
])

review_schema = StructType([
    StructField("review_id", StringType()),
    StructField("order_id", StringType()),
    StructField("review_score", IntegerType()),
    StructField("review_comment_title", StringType()),
    StructField("review_comment_message", StringType()),
    StructField("review_creation_date", TimestampType()),
    StructField("review_answer_timestamp", TimestampType())
])

geolocation_schema = StructType([
    StructField("geolocation_zip_code_prefix", StringType()),
    StructField("geolocation_lat", DoubleType()),
    StructField("geolocation_lng", DoubleType()),
    StructField("geolocation_city", StringType()),
    StructField("geolocation_state", StringType())
])

category_translation_schema = StructType([
    StructField("product_category_name", StringType()),
    StructField("product_category_name_english", StringType())
])


## Project modules


In [134]:
import sys
import logging

sys.path.append("../src")

from ingestion import ingest_olist_data
from validation import (
    null_checks, duplicate_checks, date_validation,
    valid_values_check, referential_integrity_check
)
from transformation import add_delivery_days, add_date_parts, add_late_delivery_flag
from curation import aggregate_order_items, aggregate_payment, build_order_curated


## Ingestion configuration


In [135]:
olist_config = {
    "orders": {"filename": "olist_orders_dataset.csv", "schema": order_schema},
    "customers": {"filename": "olist_customers_dataset.csv", "schema": customer_schema},
    "geolocation": {"filename": "olist_geolocation_dataset.csv", "schema": geolocation_schema},
    "order_items": {"filename": "olist_order_items_dataset.csv", "schema": order_item_schema},
    "order_payment": {"filename": "olist_order_payments_dataset.csv", "schema": payment_schema},
    "orders_reviews": {"filename": "olist_order_reviews_dataset.csv", "schema": review_schema},
    "products": {"filename": "olist_products_dataset.csv", "schema": product_schema},
    "sellers": {"filename": "olist_sellers_dataset.csv", "schema": seller_schema},
    "category_name_translation": {
        "filename": "product_category_name_translation.csv",
        "schema": category_translation_schema
    }
}


## Ingest all datasets


In [136]:
logging.basicConfig(level=logging.INFO)

dataframe = {}

for name, config in olist_config.items():
    try:
        dataframe[name] = ingest_olist_data(
            spark=spark,
            raw_path_to_file=config["filename"],
            file_schema=config["schema"]
        )
    except Exception as e:
        print(f"Failed to ingest {config['filename']}: {e}")

for name, df in dataframe.items():
    print(f"{name}: {df.count()}")


INFO:ingestion:Reading file :olist_orders_dataset.csv
INFO:ingestion:Successfully read file: olist_orders_dataset.csv
INFO:ingestion:Reading file :olist_customers_dataset.csv
INFO:ingestion:Successfully read file: olist_customers_dataset.csv
INFO:ingestion:Reading file :olist_geolocation_dataset.csv
INFO:ingestion:Successfully read file: olist_geolocation_dataset.csv
INFO:ingestion:Reading file :olist_order_items_dataset.csv
INFO:ingestion:Successfully read file: olist_order_items_dataset.csv
INFO:ingestion:Reading file :olist_order_payments_dataset.csv
INFO:ingestion:Successfully read file: olist_order_payments_dataset.csv
INFO:ingestion:Reading file :olist_order_reviews_dataset.csv
INFO:ingestion:Successfully read file: olist_order_reviews_dataset.csv
INFO:ingestion:Reading file :olist_products_dataset.csv
INFO:ingestion:Successfully read file: olist_products_dataset.csv
INFO:ingestion:Reading file :olist_sellers_dataset.csv
INFO:ingestion:Successfully read file: olist_sellers_datase

orders: 99441
customers: 99441
geolocation: 1000163
order_items: 112650
order_payment: 103886
orders_reviews: 104162
products: 32951
sellers: 3095
category_name_translation: 71


## Validate orders


In [137]:
valid_status = [
    "shipped", "canceled", "invoiced", "created",
    "delivered", "unavailable", "processing", "approved"
]

null_checks(dataframe["orders"]).show()

print("Duplicate order IDs:", duplicate_checks(dataframe["orders"], "order_id").count())

print(
    "Invalid delivery sequence:",
    date_validation(
        dataframe["orders"],
        "order_delivered_carrier_date",
        "order_delivered_customer_date"
    ).count()
)

print(
    "Invalid order statuses:",
    valid_values_check(dataframe["orders"], "order_status", valid_status).count()
)

print(
    "Orders with missing customers:",
    referential_integrity_check(
        dataframe["orders"],
        dataframe["customers"],
        "customer_id"
    ).count()
)

print(
    "Order items with missing orders:",
    referential_integrity_check(
        dataframe["order_items"],
        dataframe["orders"],
        "order_id"
    ).count()
)


+-------------------+----------------------+-----------------------+-----------------------------------+----------------------------+---------------------------------------+----------------------------------------+----------------------------------------+
|order_id_null_count|customer_id_null_count|order_status_null_count|order_purchase_timestamp_null_count|order_approved_at_null_count|order_delivered_carrier_date_null_count|order_delivered_customer_date_null_count|order_estimated_delivery_date_null_count|
+-------------------+----------------------+-----------------------+-----------------------------------+----------------------------+---------------------------------------+----------------------------------------+----------------------------------------+
|                  0|                     0|                      0|                                  0|                         160|                                   1783|                                    2965|                  

## Transform orders


In [138]:
orders_transformed = add_delivery_days(dataframe["orders"])
orders_transformed = add_date_parts(
    orders_transformed,
    "order_purchase_timestamp",
    "purchase"
)
orders_transformed = add_late_delivery_flag(orders_transformed)

orders_transformed.select(
    "order_id",
    "delivery_days",
    "purchase_year",
    "purchase_month",
    "is_late"
).show(5)


+--------------------+-------------+-------------+--------------+-------+
|            order_id|delivery_days|purchase_year|purchase_month|is_late|
+--------------------+-------------+-------------+--------------+-------+
|e481f51cbdc54678b...|            8|         2017|            10|      0|
|53cdb2fc8bc7dce0b...|           14|         2018|             7|      0|
|47770eb9100c2d0c4...|            9|         2018|             8|      0|
|949d5b44dbf5de918...|           14|         2017|            11|      0|
|ad21c59c0840e6cb8...|            3|         2018|             2|      0|
+--------------------+-------------+-------------+--------------+-------+
only showing top 5 rows


## Curate order-level dataset


In [139]:
order_items_agg = aggregate_order_items(dataframe["order_items"])
payment_agg = aggregate_payment(dataframe["order_payment"])

curated_orders = build_order_curated(
    orders_transformed,
    order_items_agg,
    payment_agg,
    dataframe["customers"]
)


## Final checks


In [140]:
print("Curated order rows:", curated_orders.count())
print("Duplicate order IDs:", duplicate_checks(curated_orders, "order_id").count())

curated_orders.printSchema()
curated_orders.show(5, truncate=False)


Curated order rows: 99441
Duplicate order IDs: 0
root
 |-- customer_id: string (nullable = true)
 |-- order_id: string (nullable = true)
 |-- order_status: string (nullable = true)
 |-- order_purchase_timestamp: timestamp (nullable = true)
 |-- order_approved_at: timestamp (nullable = true)
 |-- order_delivered_carrier_date: timestamp (nullable = true)
 |-- order_delivered_customer_date: timestamp (nullable = true)
 |-- order_estimated_delivery_date: timestamp (nullable = true)
 |-- delivery_days: integer (nullable = true)
 |-- purchase_year: integer (nullable = true)
 |-- purchase_month: integer (nullable = true)
 |-- is_late: integer (nullable = true)
 |-- order_item_count: long (nullable = true)
 |-- total_item_value: double (nullable = true)
 |-- total_freight: double (nullable = true)
 |-- total_payment_value: decimal(28,2) (nullable = true)
 |-- payment_count: long (nullable = true)
 |-- customer_unique_id: string (nullable = true)
 |-- customer_zip_code_prefix: string (nullable 